# 03 · 代码体检：ruff、mypy、pytest、pre-commit（路线第 1 步第 2 块）

**JD 对应**：「代码规范与质量意识」「熟悉单元测试、静态检查、CI/CD」。

## 做了什么

```bash
uv add --dev ruff mypy pytest pre-commit        # 开发依赖：开发时要用，产品运行时不需要
uv run pre-commit install --hook-type pre-commit --hook-type pre-push   # 把门的挂到 git 上
```

新增三样东西：

| 文件 | 作用 |
|---|---|
| `pyproject.toml` 里的 `[tool.ruff]` `[tool.mypy]` `[tool.pytest.ini_options]` | 三个工具的配置，集中在一份文件里 |
| `tests/test_cli.py` | 第一条测试："跑入口命令，应该打印 Hello from raredx!" |
| `.pre-commit-config.yaml` | 把门的清单：提交前跑哪些检查，推送前跑哪些 |

三道体检，一个把门的：

| 工具 | 查什么 | 像实验室里的 |
|---|---|---|
| ruff | 格式统一、没用的 import、低级错误 | 记录本字迹规范，不许涂改 |
| mypy | 类型：函数说要个整数，你给了字符串，跑之前就指出来 | 试剂标签和用途对不上就报警 |
| pytest | 跑测试：每个功能配一条"输入这个应该得到那个" | 阳性对照、阴性对照 |
| pre-commit | 提交前自动跑上面的检查，不过就提交不进去 | 不过 QC 的样本不入库 |

## 工业界叫什么

| 术语 | 大白话 |
|---|---|
| linter / formatter（ruff） | 查代码风格和低级错误的工具 / 自动排版的工具。ruff 一个顶过去的 flake8 + isort + black |
| static type checking（mypy） | 不运行程序，只读代码就能发现类型对不上的地方。同类：pyright |
| unit test（pytest） | 单元测试：对一个函数的一条"输入→期望输出"。`tests/` 目录里每个 `test_*.py` |
| dev dependency（开发依赖） | 只在开发机上装的工具，不进产品；`pyproject.toml` 的 `[dependency-groups] dev` |
| git hook / pre-commit | git 在"提交"前自动执行的脚本。pre-commit 是管理这些脚本的工具 |
| fail fast（快速失败） | 错误越早暴露越便宜：编辑器里 < 提交时 < CI 里 < 上线后 |

## 为什么

- **规矩由机器执行，不靠记性。** 旧仓库的 CI 只检查 otovce_v2 一个目录，mypy 只检查其中一个子目录，
  因为后补检查时旧代码已经过不了，只好缩小范围。新仓库第一天就开严格模式（`strict = true`），
  代码还只有两行，没有历史包袱。
- **提交前跑秒级检查，推送前跑测试。** 提交是存草稿，要快；测试会越来越慢，放在推送（交作业）前。
  钩子配置里 `stages: [pre-push]` 就是这个意思。
- **mypy 从项目环境里跑。** 钩子写的是 `uv run mypy src tests`，而不是用 pre-commit 自带的 mypy 镜像——
  后者在一个空环境里跑，看不到我们的依赖，会误报。

## 它真的拦人吗——一次真实的拒绝

我故意写了一个坏文件 `src/raredx/bad.py`：多一个没用的 `import os`，函数说返回整数却返回了 `a + "1"`。
然后 `git add` + `git commit`，把门的输出如下（原样）：

```
ruff check...............................................................Failed
- hook id: ruff-check
- files were modified by this hook
Found 1 error (1 fixed, 0 remaining).

ruff format..............................................................Failed
- hook id: ruff-format
- files were modified by this hook
1 file reformatted, 1 file left unchanged

mypy.....................................................................Failed
- hook id: mypy
- exit code: 1
src/raredx/bad.py:2: error: Unsupported operand types for + ("int" and "str")  [operator]
Found 1 error in 1 file (checked 3 source files)
```

提交被拒，记录本还是原来那几条。ruff 顺手把没用的 import 删了（"1 fixed"），mypy 指出了 `int + str`。
之后我把坏文件删掉了。

下面这格是安全版的重演：造一个坏文件，只跑检查，不尝试提交，最后把文件删掉。可以反复运行。

In [1]:
import os, subprocess
ROOT = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(ROOT)
print("当前目录:", ROOT)

当前目录: /Users/pipi/Projects/QAFI_Paper/hearing_agent/rare-disease-agent


In [2]:
bad = '''import os


def add(a: int, b: int) -> int:
    return a + "1"
'''
open("src/raredx/bad.py", "w").write(bad)
print("写入 src/raredx/bad.py，跑检查：\n")
!uv run ruff check --no-fix src/raredx/bad.py
!uv run mypy src/raredx/bad.py
os.remove("src/raredx/bad.py")
print("\n已删除 src/raredx/bad.py")

写入 src/raredx/bad.py，跑检查：



F401 [*] `os` imported but unused
 --> src/raredx/bad.py:1:8
  |
1 | import os
  |        ^^
help: Remove unused import: `os`
  |
  - import os
1 |
  |

Found 1 error.
[*] 1 fixable with the `--fix` option.


src/raredx/bad.py:5: error: Unsupported operand types for + ("int" and "str")  [operator]
Found 1 error in 1 file (checked 1 source file)



已删除 src/raredx/bad.py


## ✏️ 你来验证

三道体检各跑一遍，都应该是通过：

In [3]:
!uv run ruff check && uv run ruff format --check

All checks passed!


6 files already formatted


In [4]:
!uv run mypy src tests

Success: no issues found in 2 source files


In [5]:
!uv run pytest

.                                                                        [100%]
1 passed in 0.00s


把门的确实挂在 git 上了（`.git/hooks/` 里有 `pre-commit` 和 `pre-push`，不带 `.sample` 后缀的才是生效的）：

In [6]:
!ls .git/hooks | grep -v sample

pre-commit
pre-push


一条命令跑全部钩子：

In [7]:
!uv run pre-commit run --all-files

trim trailing whitespace.................................................

Passed
fix end of files.........................................................

Passed
check yaml...............................................................

Passed
check toml...............................................................

Passed
check for merge conflicts................................................

...

Passed


detect private key.......................................................Passed
ruff check...............................................................

Passed
ruff format..............................................................

Passed
mypy.....................................................................

Passed


## 思考

为什么 pytest 放在"推送前"而不是"提交前"？先想，再看下一格。

**答案**：提交要快（秒级），人才愿意小步频繁提交；测试会随项目变慢（分钟级），每次提交都等它，
人就会攒着不提交，又回到旧仓库的毛病。推送是"交作业"，那时跑测试刚好。真正的最后一道关是
CI（下一块）：不管本地钩子装没装，服务器上都会再跑一遍全部检查。